<a href="https://colab.research.google.com/github/deekshu15/CNN_KWS/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [64]:
import sys
sys.path.append("/content/CNN_KWS_PROJECT")

In [70]:
!cd /content
!rm -rf CNN_KWS
!git clone https://github.com/deekshu15/CNN_KWS.git
!cd CNN_KWS

Cloning into 'CNN_KWS'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 274 (delta 14), reused 32 (delta 7), pack-reused 230 (from 1)
Receiving objects: 100% (274/274), 3.07 MiB | 5.03 MiB/s, done.
Resolving deltas: 100% (112/112), done.


In [7]:
!pip install torch torchaudio soundfile pandas numpy tqdm

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [9]:
import pandas as pd
import os

# -------- CONFIG --------
GLOBAL_META = "/content/drive/MyDrive/KWS_DATA/metadata_fixed.csv"
OUT_ROOT = "/content"
AUDIO_ROOT = "/content/audio"
NUM_FOLDERS = 12
# ------------------------

df = pd.read_csv(GLOBAL_META)

print("Total samples in global metadata:", len(df))

for folder_id in range(1, NUM_FOLDERS + 1):
    print(f"\n📁 Processing folder {folder_id}")

    # filter rows belonging to this folder
    mask = df["audio_path"].astype(str).str.contains(f"folder {folder_id}")
    df_f = df[mask].copy()

    if len(df_f) == 0:
        print(f"⚠️ No samples found for folder {folder_id}, skipping")
        continue

    # rewrite audio_path to Colab-friendly path
    df_f["audio_path"] = df_f["audio_path"].apply(
        lambda p: f"{AUDIO_ROOT}/folder_{folder_id}/" + os.path.basename(str(p))
    )

    out_csv = f"{OUT_ROOT}/metadata_folder{folder_id}_fixed.csv"
    df_f.to_csv(out_csv, index=False)

    print(f"✅ Saved {out_csv} | samples: {len(df_f)}")

Total samples in global metadata: 199010

📁 Processing folder 1
✅ Saved /content/metadata_folder1_fixed.csv | samples: 62972

📁 Processing folder 2
✅ Saved /content/metadata_folder2_fixed.csv | samples: 17962

📁 Processing folder 3
✅ Saved /content/metadata_folder3_fixed.csv | samples: 17471

📁 Processing folder 4
✅ Saved /content/metadata_folder4_fixed.csv | samples: 18109

📁 Processing folder 5
✅ Saved /content/metadata_folder5_fixed.csv | samples: 15924

📁 Processing folder 6
✅ Saved /content/metadata_folder6_fixed.csv | samples: 14986

📁 Processing folder 7
✅ Saved /content/metadata_folder7_fixed.csv | samples: 15995

📁 Processing folder 8
✅ Saved /content/metadata_folder8_fixed.csv | samples: 17587

📁 Processing folder 9
✅ Saved /content/metadata_folder9_fixed.csv | samples: 18004

📁 Processing folder 10
✅ Saved /content/metadata_folder10_fixed.csv | samples: 17912

📁 Processing folder 11
✅ Saved /content/metadata_folder11_fixed.csv | samples: 18130

📁 Processing folder 12
✅ Saved

In [10]:
import pandas as pd

df = pd.read_csv("/content/metadata_folder1_fixed.csv")

chars = set()
for kw in df["keyword"]:
    chars.update(list(str(kw)))

char2idx = {c: i + 1 for i, c in enumerate(sorted(chars))}
char2idx["<PAD>"] = 0

In [11]:
import os
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

from CNN_KWS.datasets.kws_dataset import KWSDataset
from CNN_KWS.utils.collate import collate
from CNN_KWS.models.kws_model import KWSModel
from CNN_KWS.train.train_incremental import train_one_folder

In [12]:
from torch.utils.data import DataLoader

ds = KWSDataset(
    metadata_csv="/content/metadata_folder1_fixed.csv",
    char2idx=char2idx
)

loader = DataLoader(
    ds,
    batch_size=4,
    shuffle=True,
    collate_fn=collate
)

Loaded samples: 62972


In [13]:
import pandas as pd

for i in range(1, 13):
    path = f"/content/metadata_folder{i}_fixed.csv"
    df = pd.read_csv(path)
    print(f"Folder {i}: {len(df)} samples")

Folder 1: 62972 samples
Folder 2: 17962 samples
Folder 3: 17471 samples
Folder 4: 18109 samples
Folder 5: 15924 samples
Folder 6: 14986 samples
Folder 7: 15995 samples
Folder 8: 17587 samples
Folder 9: 18004 samples
Folder 10: 17912 samples
Folder 11: 18130 samples
Folder 12: 17822 samples


In [14]:
!mkdir -p /content/audio

In [15]:
!for i in 1 2 3 4 5 6 7 8 9 10 11 12; do \
    echo "📦 Extracting folder $i"; \
    mkdir -p "/content/audio/folder_$i"; \
    unzip -oq "/content/drive/MyDrive/KWS_DATA/folder $i.zip" -d "/content/audio/folder_$i"; \
done

📦 Extracting folder 1
📦 Extracting folder 2
📦 Extracting folder 3
📦 Extracting folder 4
📦 Extracting folder 5
📦 Extracting folder 6
📦 Extracting folder 7
📦 Extracting folder 8
📦 Extracting folder 9
📦 Extracting folder 10
📦 Extracting folder 11
📦 Extracting folder 12


In [16]:
import pandas as pd
import os

AUDIO_ROOT = "/content/audio"
NUM_FOLDERS = 12

for folder_id in range(1, NUM_FOLDERS + 1):
    meta_path = f"/content/metadata_folder{folder_id}_fixed.csv"
    audio_dir = f"{AUDIO_ROOT}/folder_{folder_id}"

    if not os.path.exists(meta_path) or not os.path.exists(audio_dir):
        continue

    print(f"\n🔍 Resolving paths for folder {folder_id}")

    df = pd.read_csv(meta_path)
    resolved_paths = []
    missing = 0

    # Build an index of all wav files in this folder (fast)
    wav_index = {}
    for root, _, files in os.walk(audio_dir):
        for f in files:
            if f.lower().endswith(".wav"):
                wav_index[f] = os.path.join(root, f)

    for p in df["audio_path"]:
        fname = os.path.basename(str(p))
        found = wav_index.get(fname)

        if found is None:
            resolved_paths.append(None)
            missing += 1
        else:
            resolved_paths.append(found)

    df["audio_path"] = resolved_paths
    before = len(df)
    df = df.dropna(subset=["audio_path"]).reset_index(drop=True)
    after = len(df)

    df.to_csv(meta_path, index=False)
    print(f"✅ Kept {after}/{before} samples | Missing: {missing}")

print("\n🎉 All metadata paths resolved to nested audio")



🔍 Resolving paths for folder 1
✅ Kept 0/62972 samples | Missing: 62972

🔍 Resolving paths for folder 2
✅ Kept 0/17962 samples | Missing: 17962

🔍 Resolving paths for folder 3
✅ Kept 0/17471 samples | Missing: 17471

🔍 Resolving paths for folder 4
✅ Kept 0/18109 samples | Missing: 18109

🔍 Resolving paths for folder 5
✅ Kept 0/15924 samples | Missing: 15924

🔍 Resolving paths for folder 6
✅ Kept 0/14986 samples | Missing: 14986

🔍 Resolving paths for folder 7
✅ Kept 0/15995 samples | Missing: 15995

🔍 Resolving paths for folder 8
✅ Kept 0/17587 samples | Missing: 17587

🔍 Resolving paths for folder 9
✅ Kept 0/18004 samples | Missing: 18004

🔍 Resolving paths for folder 10
✅ Kept 0/17912 samples | Missing: 17912

🔍 Resolving paths for folder 11
✅ Kept 0/18130 samples | Missing: 18130

🔍 Resolving paths for folder 12
✅ Kept 0/17822 samples | Missing: 17822

🎉 All metadata paths resolved to nested audio


In [17]:
import os

AUDIO_ROOT = "/content/audio"
audio_index = {}

for folder in os.listdir(AUDIO_ROOT):
    folder_path = os.path.join(AUDIO_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue

    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".wav"):
                parts = root.split(os.sep)
                if "DATASETS" in parts:
                    speaker_id = parts[parts.index("DATASETS") - 1]
                    # One wav per speaker session is enough
                    audio_index.setdefault(speaker_id, os.path.join(root, f))

print("Indexed speakers:", len(audio_index))


Indexed speakers: 46


In [18]:
import pandas as pd
import os

META_PATH = "/content/drive/MyDrive/KWS_DATA/metadata_fixed.csv"
OUT_ROOT = "/content"
NUM_FOLDERS = 12

df = pd.read_csv(META_PATH)
print("Loaded metadata rows:", len(df))

# Keep only rows whose speaker_id has audio
aligned_df = df[df["speaker_id"].isin(audio_index.keys())].copy()
print("Rows with audio:", len(aligned_df))

# Replace audio_path with real Colab path
aligned_df["audio_path"] = aligned_df["speaker_id"].apply(
    lambda sid: audio_index[sid]
)

# Split into folder-wise metadata
for folder_id in range(1, NUM_FOLDERS + 1):
    mask = df["audio_path"].astype(str).str.contains(f"folder {folder_id}")
    df_f = aligned_df[mask].copy()

    out_csv = f"{OUT_ROOT}/metadata_folder{folder_id}_aligned.csv"
    df_f.to_csv(out_csv, index=False)

    print(f"Folder {folder_id}: {len(df_f)} samples saved")


Loaded metadata rows: 199010
Rows with audio: 199010
Folder 1: 62972 samples saved
Folder 2: 17962 samples saved
Folder 3: 17471 samples saved
Folder 4: 18109 samples saved
Folder 5: 15924 samples saved
Folder 6: 14986 samples saved
Folder 7: 15995 samples saved
Folder 8: 17587 samples saved
Folder 9: 18004 samples saved
Folder 10: 17912 samples saved
Folder 11: 18130 samples saved
Folder 12: 17822 samples saved


In [19]:
import torch
from CNN_KWS.models.kws_model import KWSModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = KWSModel(len(char2idx)).to(DEVICE)
print("Model created on", DEVICE)

Model created on cuda


In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Optimizer created")

Optimizer created


In [21]:
from CNN_KWS.datasets.kws_dataset import KWSDataset

ds = KWSDataset(
    metadata_csv="/content/metadata_folder1_aligned.csv",
    char2idx=char2idx
)

print(len(ds))

Loaded samples: 62972
62972


In [22]:
from CNN_KWS.train.train_incremental import train_one_folder

In [23]:
from CNN_KWS.datasets.kws_dataset import KWSDataset

ds = KWSDataset(
    metadata_csv="/content/metadata_folder1_aligned.csv",
    char2idx=char2idx
)
print("Samples:", len(ds))

Loaded samples: 62972
Samples: 62972


In [24]:
from CNN_KWS.datasets.kws_dataset import KWSDataset
from CNN_KWS.utils.collate import collate
from torch.utils.data import DataLoader

ds = KWSDataset(
    metadata_csv="/content/metadata_folder1_aligned.csv",
    char2idx=char2idx
)

loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate)

m, k, kl, y, mask = next(iter(loader))
print("m:", m.shape)
print("k:", k.shape, "kl:", kl)
print("y max:", y.max().item())

Loaded samples: 62972
m: torch.Size([4, 1001, 80])
k: torch.Size([4, 5]) kl: tensor([4, 2, 5, 5])
y max: 1.0


In [25]:
import os, pandas as pd

path = "/content/metadata_folder1_aligned.csv"
print("Exists:", os.path.exists(path))
df = pd.read_csv(path)
print("Rows:", len(df))

Exists: True
Rows: 62972


In [26]:
!grep "metadata_folder" -n CNN_KWS/train/train_incremental.py

24:    meta_csv = f"{meta_root}/metadata_folder{folder_id}_aligned.csv"


In [27]:
from CNN_KWS.train.train_incremental import train_one_folder

In [28]:
import importlib

import CNN_KWS.datasets.kws_dataset as kd
import CNN_KWS.train.train_incremental as ti

importlib.reload(kd)
importlib.reload(ti)

KWSDataset = kd.KWSDataset
train_one_folder = ti.train_one_folder

In [29]:
ds = KWSDataset(
    metadata_csv="/content/metadata_folder1_aligned.csv",
    char2idx=char2idx
)
print("Dataset length:", len(ds))

Loaded samples: 62972
Dataset length: 62972


In [72]:
final_path = "/content/drive/MyDrive/KWS_CHECKPOINTS/kws_final.pt"
torch.save(model.state_dict(), final_path)
print("✅ Final model saved:", final_path)

✅ Final model saved: /content/drive/MyDrive/KWS_CHECKPOINTS/kws_final.pt


In [31]:
from CNN_KWS.inference.inference import KWSInferencer

In [32]:
import importlib
import CNN_KWS.inference.inference as inf

importlib.reload(inf)

KWSInferencer = inf.KWSInferencer

In [ ]:
for folder_id in range(1, 13):

    if folder_id == 1:
        epochs = 5   # 🔥 strong base training
    else:
        epochs = 2    # light adaptation

    print(f"\n🚀 Training Folder {folder_id} for {epochs} epochs")

    model = train_one_folder(
        folder_id=folder_id,
        model=model,
        optimizer=optimizer,
        char2idx=char2idx,
        collate_fn=collate,
        epochs=epochs,
        batch_size=8,
        device=DEVICE,
        meta_root="/content"
    )


🚀 Training Folder 1 for 5 epochs

📁 Training folder 1
Loaded samples: 62972


100%|██████████| 7872/7872 [05:42<00:00, 22.99it/s]


Epoch 1 | Avg Loss: 3.015005


100%|██████████| 7872/7872 [05:36<00:00, 23.37it/s]


Epoch 2 | Avg Loss: 3.015081


...

Epoch 2 | Avg Loss: 0.966837
✅ Saved checkpoint: /content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder11.pt

🚀 Training Folder 12 for 2 epochs

📁 Training folder 12
Loaded samples: 17822


100%|██████████| 2228/2228 [01:32<00:00, 24.07it/s]


Epoch 1 | Avg Loss: 9.247821


100%|██████████| 2228/2228 [01:33<00:00, 23.81it/s]

Epoch 2 | Avg Loss: 9.245732
✅ Saved checkpoint: /content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder12.pt


In [33]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder1_epoch7.pt"
)
print("✅ Folder 1 model saved at epoch 7")

✅ Folder 1 model saved at epoch 7


In [34]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = KWSModel(len(char2idx)).to(DEVICE)
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder1_epoch7.pt",
        map_location=DEVICE
    )
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("✅ Loaded Folder 1 model (epoch 7)")

✅ Loaded Folder 1 model (epoch 7)


In [35]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = KWSModel(len(char2idx)).to(DEVICE)
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder12.pt",
        map_location=DEVICE
    )
)

model.eval()
print("✅ Final model (Folder 12) loaded")

✅ Final model (Folder 12) loaded


In [36]:
import importlib
import CNN_KWS.inference.inference as inf
importlib.reload(inf)

KWSInferencer = inf.KWSInferencer
print("✅ KWSInferencer class found")

✅ KWSInferencer class found


In [37]:
import importlib
import CNN_KWS.inference.inference as inf
importlib.reload(inf)

KWSInferencer = inf.KWSInferencer
print("✅ Inference module ready")

✅ Inference module ready


In [38]:
import pandas as pd
import numpy as np
from tqdm import tqdm


def compute_iou(gt_s, gt_e, pr_s, pr_e):
    inter = max(0.0, min(gt_e, pr_e) - max(gt_s, pr_s))
    union = max(gt_e, pr_e) - min(gt_s, pr_s)
    return inter / union if union > 0 else 0.0


def coverage(gt_s, gt_e, pr_s, pr_e):
    inter = max(0.0, min(gt_e, pr_e) - max(gt_s, pr_s))
    return inter / (gt_e - gt_s) if gt_e > gt_s else 0.0


def evaluate_folder(
    inferencer,
    metadata_csv,
    max_samples=None
):
    df = pd.read_csv(metadata_csv)

    if len(df) == 0:
        raise ValueError("Metadata file is empty")

    start_err, end_err, mae = [], [], []
    ious, coverages = [], []
    missed = 0
    total = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        wav = row["audio_path"]
        kw = row["keyword"]
        gt_s = float(row["start_time"])
        gt_e = float(row["end_time"])

        pred = inferencer.infer(wav, kw)
        total += 1

        if pred is None:
            missed += 1
            continue

        ps, pe, conf = pred

        se = abs(ps - gt_s)
        ee = abs(pe - gt_e)

        start_err.append(se)
        end_err.append(ee)
        mae.append((se + ee) / 2)

        ious.append(compute_iou(gt_s, gt_e, ps, pe))
        coverages.append(coverage(gt_s, gt_e, ps, pe))

        if max_samples and total >= max_samples:
            break

    return {
        "samples": total,
        "miss_rate_%": round(missed / total * 100, 2),
        "mean_start_error_sec": round(np.mean(start_err), 3),
        "mean_end_error_sec": round(np.mean(end_err), 3),
        "mean_mae_sec": round(np.mean(mae), 3),
        "mean_iou": round(np.mean(ious), 3),
        "mean_coverage": round(np.mean(coverages), 3),
    }


In [39]:
import pickle

with open("/content/char2idx.pkl", "wb") as f:
    pickle.dump(ds.char2idx, f)

In [40]:
print("Missing chars:", [c for c in "FRIGHTEN" if c not in char2idx])

Missing chars: []


In [41]:
from CNN_KWS.models.kws_model import KWSModel
import torch

model = KWSModel(len(char2idx))
state = torch.load("/content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder12.pt", map_location="cpu")

missing, unexpected = model.load_state_dict(state, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


In [42]:
import CNN_KWS.inference.inference as inf
print(inf.__file__)

/content/CNN_KWS/inference/inference.py


In [43]:
import importlib
import CNN_KWS.inference.inference as inf
importlib.reload(inf)

KWSInferencer = inf.KWSInferencer

In [44]:
import pandas as pd
from glob import glob

def build_keyword_stats(metadata_files):
    durations = {}

    for csv in metadata_files:
        df = pd.read_csv(csv)
        for _, r in df.iterrows():
            kw = str(r.keyword)
            dur = float(r.end_time - r.start_time)
            durations.setdefault(kw, []).append(dur)

    return {k: sum(v) / len(v) for k, v in durations.items()}


metadata_files = sorted(glob("/content/metadata_folder*_aligned.csv"))
keyword_stats = build_keyword_stats(metadata_files)

print("Total keywords:", len(keyword_stats))
print(list(keyword_stats.items())[:10])

Total keywords: 1906
[('FRIGHTEN', 0.5683609824561404), ('PLAGUE', 0.41612915789473687), ('#', 2.708739144574692), ('FAIL', 0.4566836913580247), ('PREPARATION', 0.8337044444444445), ('YOU', 0.2098065721841332), ('ENJOYED', 0.5281966796116505), ('IT', 0.18920592346820808), ('THE', 0.15981473327541268), ('PROPOSAL', 0.5708026981132075)]


In [51]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os


def interval_iou(gs, ge, ps, pe):
    inter = max(0.0, min(ge, pe) - max(gs, ps))
    union = max(ge, pe) - min(gs, ps)
    return inter / union if union > 0 else 0.0


def interval_coverage(gs, ge, ps, pe):
    inter = max(0.0, min(ge, pe) - max(gs, ps))
    return inter / (ge - gs) if ge > gs else 0.0


def evaluate_kws_accuracy(
    inferencer,
    metadata_csv,
    max_samples=500
):
    df = pd.read_csv(metadata_csv)

    if len(df) == 0:
        raise ValueError("Metadata CSV is empty")

    if max_samples is not None:
        df = df.sample(n=min(max_samples, len(df)), random_state=42)

    start_errs = []
    end_errs = []
    maes = []
    ious = []
    covers = []
    missed = 0

    for _, r in tqdm(df.iterrows(), total=len(df)):
        wav = r.audio_path
        keyword = r.keyword

        if not os.path.exists(wav):
            continue

        pred = inferencer.infer(wav, keyword)

        # ---- MISS CASE ----
        if pred is None:
            missed += 1
            continue

        # ---- SAFE PARSING ----
        try:
            if isinstance(pred, dict):
                ps = float(pred["start"])
                pe = float(pred["end"])
            else:
                ps = float(pred[0])
                pe = float(pred[1])
        except Exception:
            missed += 1
            continue

        gs = float(r.start_time)
        ge = float(r.end_time)

        start_errs.append(abs(ps - gs))
        end_errs.append(abs(pe - ge))
        maes.append((abs(ps - gs) + abs(pe - ge)) / 2)

        ious.append(interval_iou(gs, ge, ps, pe))
        covers.append(interval_coverage(gs, ge, ps, pe))

    evaluated = len(start_errs)

    if evaluated == 0:
        return {
            "samples_evaluated": 0,
            "miss_rate_%": 100.0,
            "mean_start_error_sec": None,
            "mean_end_error_sec": None,
            "mean_mae_sec": None,
            "mean_iou": None,
            "mean_coverage": None,
        }

    return {
        "samples_evaluated": evaluated,
        "miss_rate_%": round(missed / len(df) * 100, 2),
        "mean_start_error_sec": round(float(np.mean(start_errs)), 3),
        "mean_end_error_sec": round(float(np.mean(end_errs)), 3),
        "mean_mae_sec": round(float(np.mean(maes)), 3),
        "mean_iou": round(float(np.mean(ious)), 3),
        "mean_coverage": round(float(np.mean(covers)), 3),
    }


In [52]:
metrics = evaluate_kws_accuracy(
    inferencer,
    metadata_csv="/content/metadata_folder1_aligned.csv",
    max_samples=500
)

metrics

100%|██████████| 500/500 [00:04<00:00, 124.83it/s]


{'samples_evaluated': 356,
 'miss_rate_%': 28.8,
 'mean_start_error_sec': 0.897,
 'mean_end_error_sec': 0.918,
 'mean_mae_sec': 0.907,
 'mean_iou': 0.152,
 'mean_coverage': 0.205}

In [53]:
import pandas as pd

summary_df = pd.DataFrame({
    "Metric": [
        "Miss Rate (%)",
        "Mean Start Error (s)",
        "Mean End Error (s)",
        "Mean MAE (s)",
        "Mean IoU",
        "Mean Coverage"
    ],
    "Value": [
        metrics["miss_rate_%"],
        metrics["mean_start_error_sec"],
        metrics["mean_end_error_sec"],
        metrics["mean_mae_sec"],
        metrics["mean_iou"],
        metrics["mean_coverage"]
    ]
})

summary_df

,Metric,Value
0,Miss Rate (%),28.800
1,Mean Start Error (s),0.897
2,Mean End Error (s),0.918
3,Mean MAE (s),0.907
4,Mean IoU,0.152
5,Mean Coverage,0.205


In [69]:
from CNN_KWS.inference.inference import KWSInferencer

inferencer = KWSInferencer(
    checkpoint_path="/content/drive/MyDrive/KWS_CHECKPOINTS/kws_folder12.pt",
    char2idx=char2idx,
    keyword_stats=keyword_stats,
    device="cpu"
)
wav = "/content/audio/folder_7/folder 7/Gaytri_VRZs68xVG1allpy7DxLKArqkeK13/DATASETS/00JS.wav"

print(inferencer.infer(wav, "PLAGUE"))

{'start': 0.64, 'end': 1.04}
